# NaN Edge Cases: Legacy vs v1

Development notebook investigating how NaN behaves across linopy operations under both conventions.

**Core principle (v1):** NaN means "absent term" — not a numeric value. It enters only through structural operations (`shift`, `where`, `reindex`, `mask=`) and propagates via IEEE semantics. Absent terms don't poison valid terms at the same coordinate.

1. [Sources of NaN](#sources-of-nan)
2. [isnull detection](#isnull-detection)
3. [Arithmetic on shifted expressions](#arithmetic-on-shifted-expressions)
4. [Combining expressions with absent terms](#combining-expressions-with-absent-terms)
5. [Constraints from expressions with NaN](#constraints-from-expressions-with-nan)
6. [Why fillna on expressions is unnecessary](#why-fillna-on-expressions-is-unnecessary)
7. [FILL_VALUE internals](#fill_value-internals)

In [ ]:
import warnings

import pandas as pd
import xarray as xr

import linopy
from linopy import Model
from linopy.config import LinopyDeprecationWarning
from linopy.expressions import FILL_VALUE

warnings.filterwarnings("ignore", category=LinopyDeprecationWarning)

MindOpt 2.2.0 | 2e28db43, Aug 29 2025, 14:27:12 | arm64 - macOS 26.2
Start license validation (current time : 14-MAR-2026 17:44:50 UTC+0100).
[WARN ] No license file is found.
[ERROR] No valid license was found. Please visit https://opt.aliyun.com/doc/latest/en/html/installation/license.html to apply for and set up your license.
License validation terminated. Time : 0.000s



In [ ]:
def make_model():
    m = Model()
    time = pd.RangeIndex(5, name="time")
    x = m.add_variables(lower=0, coords=[time], name="x")
    return m, x

---

## Sources of NaN

### shift

`.shift()` is the primary structural source of NaN. It shifts data along a dimension, creating a gap filled with `FILL_VALUE` (`vars=-1`, `coeffs=NaN`, `const=NaN`).

In [ ]:
m, x = make_model()
expr = 2 * x + 10
expr.shift(time=1)

LinearExpression [time: 5]:
---------------------------
[0]: None
[1]: +2 x[0] + 10
[2]: +2 x[1] + 10
[3]: +2 x[2] + 10
[4]: +2 x[3] + 10

In [ ]:
# Variables also support shift — labels get -1 sentinel, bounds get NaN
x.shift(time=1)

Variable (time: 5) - 1 masked entries
-------------------------------------
[0]: None
[1]: x[0] ∈ [0, inf]
[2]: x[1] ∈ [0, inf]
[3]: x[2] ∈ [0, inf]
[4]: x[3] ∈ [0, inf]

### roll

`.roll()` is circular — values wrap around, no NaN introduced.

In [ ]:
m, x = make_model()
(2 * x + 10).roll(time=1)

LinearExpression [time: 5]:
---------------------------
[0]: +2 x[4] + 10
[1]: +2 x[0] + 10
[2]: +2 x[1] + 10
[3]: +2 x[2] + 10
[4]: +2 x[3] + 10

### where

`.where(cond)` masks slots where the condition is False → `vars=-1, coeffs=NaN, const=NaN`.

In [ ]:
m, x = make_model()
mask = xr.DataArray([True, True, False, False, True], dims=["time"])
(2 * x + 10).where(mask)

LinearExpression [time: 5]:
---------------------------
[0]: +2 x[0] + 10
[1]: +2 x[1] + 10
[2]: None
[3]: None
[4]: +2 x[4] + 10

### reindex

`.reindex()` expands or shrinks coordinates. New coordinates get `FILL_VALUE`.

In [ ]:
m, x = make_model()
expr = 2 * x + 10

# Expand to a larger index — new positions [5, 6] are absent
expr.reindex({"time": pd.RangeIndex(7, name="time")})

LinearExpression [time: 7]:
---------------------------
[0]: +2 x[0] + 10
[1]: +2 x[1] + 10
[2]: +2 x[2] + 10
[3]: +2 x[3] + 10
[4]: +2 x[4] + 10
[5]: None
[6]: None

---

## isnull detection

`isnull()` checks: `(vars == -1).all(helper_dims) & const.isnull()`

Both conditions must be true — a slot is only "absent" if there are no variables AND no constant. This distinguishes "absent" from "valid expression with zero constant".

In [ ]:
m, x = make_model()
shifted = (2 * x + 10).shift(time=2)
shifted.isnull()

<xarray.DataArray (time: 5)> Size: 5B
array([ True,  True, False, False, False])
Coordinates:
  * time     (time) int64 40B 0 1 2 3 4

---

## Arithmetic on shifted expressions

When you do arithmetic on an expression that already has NaN (from `shift`/`where`/`reindex`), the NaN is **internal** — not user-supplied.

- **Legacy**: fills expression NaN with neutral elements before operating → can **revive** absent slots
- **v1**: IEEE NaN propagation → absent stays absent

In [ ]:
linopy.options["arithmetic_convention"] = "legacy"
m, x = make_model()
shifted = (2 * x + 10).shift(time=1)

# Legacy: NaN const filled with 0, then +5 = 5. Slot looks alive!
shifted + 5

LinearExpression [time: 5]:
---------------------------
[0]: +5
[1]: +2 x[0] + 15
[2]: +2 x[1] + 15
[3]: +2 x[2] + 15
[4]: +2 x[3] + 15

In [ ]:
linopy.options["arithmetic_convention"] = "v1"
m, x = make_model()
shifted = (2 * x + 10).shift(time=1)

# v1: NaN + 5 = NaN. Absent slot stays absent.
shifted + 5

LinearExpression [time: 5]:
---------------------------
[0]: None
[1]: +2 x[0] + 15
[2]: +2 x[1] + 15
[3]: +2 x[2] + 15
[4]: +2 x[3] + 15

---

## Combining expressions with absent terms

When two expressions are merged (e.g., `x + y.shift(1)`), each term is concatenated along the `_term` dimension. The constant is summed with `skipna=True` — NaN from one operand does **not** poison the other.

**Key rule: absent terms don't poison valid terms at the same coordinate.**

In [ ]:
linopy.options["arithmetic_convention"] = "v1"
m, x = make_model()
y = m.add_variables(lower=0, coords=[pd.RangeIndex(5, name="time")], name="y")

# x is valid everywhere, y.shift(1) is absent at time=0
# → time=0 still has x's term, only y's term is absent
x + (1 * y).shift(time=1)

LinearExpression [time: 5]:
---------------------------
[0]: +1 x[0]
[1]: +1 x[1] + 1 y[0]
[2]: +1 x[2] + 1 y[1]
[3]: +1 x[3] + 1 y[2]
[4]: +1 x[4] + 1 y[3]

In [ ]:
# Shifted constant is LOST at the gap:
# (y+5).shift makes the ENTIRE expression absent at time=0 — including its constant.
# Only the outer +5 survives. time=1..4 get const=10 (shifted 5 + outer 5).
x + (1 * y + 5).shift(time=1) + 5

LinearExpression [time: 5]:
---------------------------
[0]: +1 x[0] + 5
[1]: +1 x[1] + 1 y[0] + 10
[2]: +1 x[2] + 1 y[1] + 10
[3]: +1 x[3] + 1 y[2] + 10
[4]: +1 x[4] + 1 y[3] + 10

In [ ]:
# Both expressions shifted — all variable terms absent at time=0, but const=0 (not NaN)
# because merge sums constants with fill_value=0. So isnull is False — it's a zero expression, not absent.
result = (1 * x).shift(time=1) + (1 * y).shift(time=1)
print("isnull:", result.isnull().values)
result

isnull: [False False False False False]


LinearExpression [time: 5]:
---------------------------
[0]: +0
[1]: +1 x[0] + 1 y[0]
[2]: +1 x[1] + 1 y[1]
[3]: +1 x[2] + 1 y[2]
[4]: +1 x[3] + 1 y[3]

### Summary

| Expression | const at time=0 | isnull at time=0 | Why |
|---|---|---|---|
| `x + y.shift(1)` | 0 | False | y's term absent, x valid, const sum skips NaN |
| `x + y.shift(1) + 5` | 5 | False | Same, then +5 on const |
| `x + (y+5).shift(1) + 5` | 5 | False | Shifted const (5) is lost — only outer +5 survives |
| `x.shift(1) + y.shift(1)` | 0 | False | All terms absent, but const=0 (merge sums with fill_value=0) |

Note: `merge()` sums constants with `fill_value=0`, so combining two fully-absent expressions yields a zero expression (const=0), not an absent one (const=NaN). This is a design choice — the slot has no variables but a valid constant of 0.

### Legacy vs v1: scalar arithmetic on shifted expressions

| | Legacy | v1 |
|---|---|---|
| `shifted + 5` at absent slot | const=5 (alive!) | const=NaN (absent) |
| `shifted * 3` at absent slot | coeffs=0, const=0 | coeffs=NaN, const=NaN |
| `isnull()` after arithmetic | False (slot revived!) | True (slot stays absent) |

Legacy can **revive** absent slots through scalar arithmetic. v1 cannot — once absent, always absent.

---

## Constraints from expressions with NaN

Absent slots in expressions propagate to constraint RHS. The preferred approach is to avoid NaN entirely using `isel` + positional alignment, or to filter with `.sel()`.

In [ ]:
linopy.options["arithmetic_convention"] = "v1"
m, x = make_model()

# Preferred: isel + override avoids NaN entirely
x_now = 1 * x.isel(time=slice(1, None))
x_prev = 1 * x.isel(time=slice(None, -1))
ramp = x_now.sub(x_prev, join="override")
ramp

LinearExpression [time: 4]:
---------------------------
[1]: +1 x[1] - 1 x[0]
[2]: +1 x[2] - 1 x[1]
[3]: +1 x[3] - 1 x[2]
[4]: +1 x[4] - 1 x[3]

In [ ]:
# Alternative: filter absent slots with .sel() after shift
shifted = (1 * x).shift(time=1)
valid = ~shifted.isnull()
shifted.sel(time=valid)

LinearExpression [time: 4]:
---------------------------
[1]: +1 x[0]
[2]: +1 x[1]
[3]: +1 x[2]
[4]: +1 x[3]

---

## Why fillna on expressions is unnecessary

A common concern: `fillna()` works on parameters (plain DataArrays), but what about Variables and Expressions? Can NaN "leak" into arithmetic?

**No — by design.** NaN in coefficients is a structural marker meaning "this term doesn't exist," not a numeric gap that needs filling.

- **Parameters**: `fillna(0)` or `fillna(1)` makes sense — these are numeric values with a context-dependent neutral element.
- **Variables**: A decision variable either exists at a coordinate or it doesn't. There's no meaningful numeric fill for an absent variable.
- **Expressions**: Absent terms (`vars=-1, coeffs=NaN`) are filtered out at solve time. They don't contribute to the constraint matrix. No fill needed.

When expressions are combined via outer join, absent terms on one side don't poison valid terms on the other — each term in the `_term` dimension is independent.

In [ ]:
linopy.options["arithmetic_convention"] = "v1"
m = Model()
tech_a = ["wind", "solar"]
tech_b = ["solar", "gas"]

cap_a = m.add_variables(lower=0, coords=[tech_a], name="cap_a")
cap_b = m.add_variables(lower=0, coords=[tech_b], name="cap_b")

cost_a = xr.DataArray([10, 20], coords=[("dim_0", tech_a)])
cost_b = xr.DataArray([15, 25], coords=[("dim_0", tech_b)])

# Outer join: absent terms at wind (no cap_b) and gas (no cap_a)
combined = (cap_a * cost_a).add(cap_b * cost_b, join="outer")
combined

LinearExpression [dim_0: 3]:
----------------------------
[gas]: +25 cap_b[gas]
[solar]: +20 cap_a[solar] + 15 cap_b[solar]
[wind]: +10 cap_a[wind]

In [ ]:
# Further arithmetic: NaN coeffs stay NaN (absent stays absent), valid terms scale correctly
# No coordinate is fully absent — isnull is False everywhere
print("isnull:", combined.isnull().values)
combined * 2

isnull: [False False False]


LinearExpression [dim_0: 3]:
----------------------------
[gas]: +50 cap_b[gas]
[solar]: +40 cap_a[solar] + 30 cap_b[solar]
[wind]: +20 cap_a[wind]

In [ ]:
# This solves correctly — absent terms are ignored in the constraint matrix
m.add_constraints(cap_a <= 100, name="max_a")
m.add_constraints(cap_b <= 100, name="max_b")
m.add_objective(combined.sum())
status, _ = m.solve("highs")
print(f"Status: {status}, Objective: {m.objective.value}")

Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-7inffcby has 4 rows; 4 cols; 4 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+01, 2e+01]
  Bound   [0e+00, 0e+00]
  RHS     [1e+02, 1e+02]
Presolving model
0 rows, 0 cols, 0 nonzeros  0s
0 rows, 0 cols, 0 nonzeros  0s
Presolve reductions: rows 0(-4); columns 0(-4); nonzeros 0(-4) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-7inffcby
Model status        : Optimal
Objective value     :  0.0000000000e+00
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00
Status: ok, Objective: 0.0


---

## FILL_VALUE internals

| Type | Field | FILL_VALUE | Why |
|---|---|---|---|
| LinearExpression | `vars` | -1 | Integer sentinel (no variable) |
| LinearExpression | `coeffs` | NaN | Absent — not a numeric value |
| LinearExpression | `const` | NaN | Absent — needed for `isnull()` detection |
| Variable | `labels` | -1 | Integer sentinel (no variable) |
| Variable | `lower` | NaN | Absent bound |
| Variable | `upper` | NaN | Absent bound |

All float fields use NaN for absence. Integer fields use -1.

In [ ]:
print("FILL_VALUE:", FILL_VALUE)

FILL_VALUE: {'vars': -1, 'coeffs': nan, 'const': nan}


In [ ]:
linopy.options.reset()